In [15]:
# shape, columns, missing

import pandas as pd
import re
from pathlib import Path

df = pd.read_csv("../data/raw/fake_job_postings.csv")

print(df.shape)
print(list(df.columns))
print(df.isna().mean().sort_values(ascending=False).round(3))

(17880, 18)
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
salary_range           0.840
department             0.646
required_education     0.453
benefits               0.403
required_experience    0.394
function               0.361
industry               0.274
employment_type        0.194
company_profile        0.185
requirements           0.151
location               0.019
description            0.000
title                  0.000
job_id                 0.000
telecommuting          0.000
has_questions          0.000
has_company_logo       0.000
fraudulent             0.000
dtype: float64


In [16]:
# class balance
print(df.fraudulent.value_counts())
print("fraud share:", round(df.fraudulent.mean(), 4))

fraudulent
0    17014
1      866
Name: count, dtype: int64
fraud share: 0.0484


In [17]:
# duplicates
key = (df.title.fillna("") + df.description.fillna("") + df.requirements.fillna(""))
key = key.str.lower().str.replace(r"\s+", "", regex=True)
print("duplicate rows:", key.duplicated().sum())
labels_per_group = df.assign(key=key).groupby("key").fraudulent.nunique()
print("duplicate groups with conflicting fraud labels:", (labels_per_group > 1).sum())

duplicate rows: 2255
duplicate groups with conflicting fraud labels: 0


In [18]:
# internship / entry-level content
print(df.required_experience.value_counts(dropna=False))
print("'intern' in title:", df.title.str.contains("intern", case=False, na=False).sum())
text = (df.title.fillna("") + " " + df.description.fillna("") + " " + df.requirements.fillna("")).str.lower()
terms = ["python", "sql", "java", "scikit", "data analyst", "machine learning",
         "data entry", "work from home", "no experience"]
for t in terms:
    print(f"{t:18s}", text.str.contains(t, regex=False).sum())

required_experience
NaN                 7050
Mid-Senior level    3809
Entry level         2697
Associate           2297
Not Applicable      1116
Director             389
Internship           381
Executive            141
Name: count, dtype: int64
'intern' in title: 581
python             642
sql                1984
java               2280
scikit             4
data analyst       69
machine learning   152
data entry         408
work from home     173
no experience      141


In [19]:
# text length
words = text.str.split().str.len()
print(words.describe())
print("90th percentile:", words.quantile(0.9))

count    17880.000000
mean       252.240828
std        153.999972
min          2.000000
25%        139.000000
50%        232.000000
75%        328.000000
max       2118.000000
dtype: float64
90th percentile: 447.0


In [20]:
# placeholder tokens and HTML
raw = (df.description.fillna("") + " " + df.requirements.fillna("") + " " +
       df.company_profile.fillna("") + " " + df.benefits.fillna(""))
for name in ["URL", "EMAIL", "PHONE"]:
    print(name, "tokens in", raw.str.contains(rf"#{name}_\w+#").sum(), "postings")
print("postings with <tags>:", df.description.fillna("").str.contains(r"<[^>]+>").sum())
print("postings with &amp;-style codes:", df.description.fillna("").str.contains(r"&\w+;").sum())

URL tokens in 5621 postings
EMAIL tokens in 1335 postings
PHONE tokens in 527 postings
postings with <tags>: 0
postings with &amp;-style codes: 3942


In [21]:
# fraud vs real
for col in ["has_company_logo", "telecommuting", "has_questions"]:
    print(col); print(df.groupby("fraudulent")[col].mean().round(3))
print("empty company_profile share:")
print(df.assign(e=df.company_profile.isna()).groupby("fraudulent").e.mean().round(3))
print("empty requirements share:")
print(df.assign(e=df.requirements.isna()).groupby("fraudulent").e.mean().round(3))
print(pd.crosstab(df.employment_type.fillna("NA"), df.fraudulent, normalize="index").round(3))

has_company_logo
fraudulent
0    0.819
1    0.327
Name: has_company_logo, dtype: float64
telecommuting
fraudulent
0    0.041
1    0.074
Name: telecommuting, dtype: float64
has_questions
fraudulent
0    0.502
1    0.289
Name: has_questions, dtype: float64
empty company_profile share:
fraudulent
0    0.160
1    0.678
Name: e, dtype: float64
empty requirements share:
fraudulent
0    0.149
1    0.178
Name: e, dtype: float64
fraudulent           0      1
employment_type              
Contract         0.971  0.029
Full-time        0.958  0.042
NA               0.931  0.069
Other            0.934  0.066
Part-time        0.907  0.093
Temporary        0.992  0.008
